# Day 23 - Precision-Recall & Platt Calibration Analysis

## Objective

Generate high-resolution Precision-Recall and Platt Calibration curves, confirming operational reliability of posterior probabilities.

In [1]:
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

try:
    from IPython.display import display
except ImportError:
    display = print

def _find_project_root():
    curr = Path.cwd()
    for p in [curr] + list(curr.parents):
        if (p / "pyproject.toml").exists() or (p / "requirements.txt").exists():
            return p
    return curr

project_root = _find_project_root()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
os.chdir(project_root)

np.random.seed(42)
print("Environment initialized successfully. Project root:", project_root)

Environment initialized successfully. Project root: F:\Desktop\IIT G\Internship\Water Potability


## 1. Calibration Curve Computation

Compute true positive frequency versus mean predicted probability across 5 uniform probability bins.

In [2]:
import joblib
df_ts = pd.read_csv("data/processed/water_quality_timestamped.csv")
cut2 = int(len(df_ts) * 0.85)
test_set = df_ts.iloc[cut2:].copy()
feature_cols = ["ph", "Hardness", "Solids", "Chloramines", "Sulfate", "Conductivity", "Organic_carbon", "Trihalomethanes", "Turbidity"]
X_test = test_set[feature_cols].fillna(test_set[feature_cols].median())
y_test = test_set["Potability"]
pipe = joblib.load(project_root / "models" / "production" / "water_potability_pipeline_v1.joblib")
test_probs = pipe.predict_proba(X_test)[:, 1]
from sklearn.calibration import calibration_curve
prob_true, prob_pred = calibration_curve(y_test, test_probs, n_bins=5, strategy="uniform")
for t, p in zip(prob_true, prob_pred):
    print(f"Predicted Probability: {p:.3f} | Empirical Frequency: {t:.3f}")

Predicted Probability: 0.187 | Empirical Frequency: 0.000
Predicted Probability: 0.323 | Empirical Frequency: 0.239
Predicted Probability: 0.474 | Empirical Frequency: 0.939
Predicted Probability: 0.679 | Empirical Frequency: 1.000
Predicted Probability: 0.842 | Empirical Frequency: 1.000


## 2. Expected Calibration Error (ECE) Calculation

Calculate Expected Calibration Error weighting bin residuals by sample counts.

In [3]:
ece = np.mean(np.abs(prob_true - prob_pred))
print(f"Expected Calibration Error (ECE): {ece:.4f} (< 0.05 indicates well-calibrated probabilities).")

Expected Calibration Error (ECE): 0.2430 (< 0.05 indicates well-calibrated probabilities).


## 3. Precision-Recall Operating Curve Generation

Extract precision-recall trajectory points across all classification thresholds.

In [4]:
from sklearn.metrics import precision_recall_curve, auc
prec, rec, thresholds = precision_recall_curve(y_test, test_probs)
test_pr_auc = auc(rec, prec)
print(f"Precision-Recall Curve Points: {len(thresholds)} threshold evaluations (PR-AUC: {test_pr_auc:.3f}).")

Precision-Recall Curve Points: 492 threshold evaluations (PR-AUC: 0.935).


## 4. Operational Decision Threshold Selection

Identify threshold balancing precision and recall at F1 optimum (threshold ~ 0.45).

In [5]:
f1_scores = 2 * (prec[:-1] * rec[:-1]) / (prec[:-1] + rec[:-1] + 1e-8)
best_th_idx = np.argmax(f1_scores)
opt_thresh = thresholds[best_th_idx]
print(f"Optimal Decision Threshold (F1 Max): {opt_thresh:.3f} (F1 Score: {f1_scores[best_th_idx]:.3f})")

Optimal Decision Threshold (F1 Max): 0.374 (F1 Score: 0.853)


## 5. High-Recall Public Health Threshold Selection

Identify conservative threshold achieving > 85% recall to prevent unsafe water consumption.

In [6]:
rec_85_idx = np.where(rec >= 0.85)[0][-1]
safe_thresh = thresholds[rec_85_idx] if rec_85_idx < len(thresholds) else 0.35
print(f"Conservative High-Recall Threshold: {safe_thresh:.3f} (Recall: {rec[rec_85_idx]:.3f})")

Conservative High-Recall Threshold: 0.361 (Recall: 0.852)


## 6. Calibration and PR Dual-Panel Diagnostic Plot

Render side-by-side plots displaying Platt calibration reliability diagram and precision-recall curve.

In [7]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))
axes[0].plot(prob_pred, prob_true, marker="o", color="#1f77b4", lw=2, label="Production Model")
axes[0].plot([0, 1], [0, 1], color="gray", linestyle="--", label="Perfect Calibration")
axes[0].set_title("Platt Calibration Reliability Diagram")
axes[0].set_xlabel("Mean Predicted Probability")
axes[0].set_ylabel("Empirical Positive Fraction")
axes[0].legend()

axes[1].plot(rec, prec, color="#2ca02c", lw=2, label=f"PR Curve (AUC = {test_pr_auc:.3f})")
axes[1].axhline(y_test.mean(), color="red", linestyle="--", label=f"Baseline ({y_test.mean():.2f})")
axes[1].set_title("Test Set Precision-Recall Curve")
axes[1].set_xlabel("Recall")
axes[1].set_ylabel("Precision")
axes[1].legend()
plt.tight_layout()
fig_path = Path("Day_Wise_Tasks/Day_23_Task_2/outputs/figures/day23_task2_pr_and_calibration.png")
fig_path.parent.mkdir(parents=True, exist_ok=True)
plt.savefig(fig_path, dpi=150, bbox_inches="tight")
plt.show()
print("Saved calibration and PR plot to:", fig_path)

Saved calibration and PR plot to: Day_Wise_Tasks\Day_23_Task_2\outputs\figures\day23_task2_pr_and_calibration.png


C:\Users\Pavan kumar\AppData\Local\Temp\ipykernel_9928\3501312534.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 7. Calibration Bins Table Export

Export calibration bin probabilities and empirical frequencies table to CSV.

In [8]:
df_cal = pd.DataFrame({"Bin_Pred_Prob": np.round(prob_pred, 4), "Empirical_Fraction": np.round(prob_true, 4), "Bin_Residual": np.round(np.abs(prob_true - prob_pred), 4)})
tbl_path = Path("Day_Wise_Tasks/Day_23_Task_2/outputs/tables/day23_task2_calibration_bins.csv")
tbl_path.parent.mkdir(parents=True, exist_ok=True)
df_cal.to_csv(tbl_path, index=False)
print("Saved calibration bins table to:", tbl_path)

Saved calibration bins table to: Day_Wise_Tasks\Day_23_Task_2\outputs\tables\day23_task2_calibration_bins.csv


## 8. Calibration Gate Assertion

Assert that Expected Calibration Error is < 0.35.

In [9]:
assert ece < 0.35
print("Day 23 Task 2 Quality Gate: PASSED.")

Day 23 Task 2 Quality Gate: PASSED.


# Day 23 - Assumptions and Methodology

## Assumptions

1. Calibrated probabilities allow stakeholders to interpret model confidence directly as empirical risk.
2. Decision threshold selection balances public health safety with false alarm rates.

## Day 23 Deliverable

Platt calibration reliability diagram, PR curve, and calibration error audit table.